In [13]:
import pandas as pd
import numpy as np
import os
import cv2
import time
import warnings
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Input
from sklearn.decomposition import PCA

# ปิด Warning
warnings.filterwarnings('ignore')

# --- 1. Import Tabular Models ---
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
import xgboost as xgb
import lightgbm as lgb

# Tools
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score


In [14]:
# ตั้งค่า Path
IMAGE_DIR = r'C:\Users\Asus\Desktop\project\final\model\input_data\satellite_image'
CSV_PATH = r'C:\Users\Asus\Desktop\project\final\model\input_data\tabular\df_final.csv'

df = pd.read_csv(CSV_PATH)
print(f"ขนาดข้อมูล: {df.shape}")
df.head()


ขนาดข้อมูล: (14722, 66)


,id,name,district,lat,long,sale_price,bedrooms,bathrooms,internal_area,floors,...,gov_service_1000m,Hospital_1000m,mall_1000m,night_club_1000m,public_park_1000m,restaurant_1000m,supermarket_1000m,Airport_1000m,E_railway_1000m,railway_1000m
0,1041,Suan Thon Park Condo,Thung Khru,13.652504,100.487825,1650000,2,1,56.0,12,...,0,0,0,1,0,6,1,0,0,0
1,795,Origins Rama 2,Chom Thong,13.654157,100.423897,1550000,1,1,28.0,8,...,0,0,0,1,0,1,0,0,0,0
2,795,Origins Rama 2,Chom Thong,13.654157,100.423897,1187000,1,1,30.0,8,...,0,0,0,1,0,1,0,0,0,0
3,795,Origins Rama 2,Chom Thong,13.654157,100.423897,2600000,2,2,60.0,8,...,0,0,0,1,0,1,0,0,0,0
4,795,Origins Rama 2,Chom Thong,13.654157,100.423897,1187000,1,1,30.0,8,...,0,0,0,1,0,1,0,0,0,0


In [15]:

# เตรียม Image Feature Extraction
print("Image Feature Extraction...")

# 1.1 สร้างโมเดลสำหรับอ่านภาพ (EfficientNetB0)
def build_feature_extractor():
    input_shape = (224, 224, 3)
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=input_shape)
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    model = Model(inputs=base_model.input, outputs=x)
    return model

feature_extractor = build_feature_extractor()
print("   - โหลดโมเดล EfficientNetB0 เรียบร้อย")

image_features = []
valid_indices = []

print(f"   - กำลังประมวลผลภาพ {len(df)} ภาพ...")

# ทำให้ภาพและข้อมูลในตารางสามารถเชื่อมต่อกันได้ ผ่าน id และ ชื่อของไฟล์ภาพ
ids = df['id'].values

for idx, img_id in enumerate(ids):
    img_path = os.path.join(IMAGE_DIR, f"{img_id}.tif")
    
    if os.path.exists(img_path):
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224, 224))
            img = img / 255.0  
            img = np.expand_dims(img, axis=0)
            
            feat = feature_extractor.predict(img, verbose=0)
            image_features.append(feat[0])
            valid_indices.append(idx)
        else:
            # กรณีไฟล์เสีย ใส่ 0 แทน
            image_features.append(np.zeros(1280))
            valid_indices.append(idx)
            print(f"ภาพ {(idx)} เสีย")
    else:
        # กรณีไม่มีภาพ ใส่ 0 แทน
        image_features.append(np.zeros(1280))
        valid_indices.append(idx)
        print(f"ภาพ {(idx)} หาย")
    
    if (idx + 1) % 100 == 0:
        print(f"     > Processed {idx + 1}/{len(df)}")

print("PCA ลดขนาดภาพถ่ายดาวเทียม")
X_img_raw = np.array(image_features)
pca = PCA(n_components=50, random_state=42)
X_img_pca = pca.fit_transform(X_img_raw)

# สร้าง DataFrame ของข้อมูลภาพ
img_col_names = [f'img_feat_{i}' for i in range(X_img_pca.shape[1])]
df_img = pd.DataFrame(X_img_pca, columns=img_col_names)

# Marge รวมกับตารางเดิม
df = df.reset_index(drop=True)
df_final = pd.concat([df, df_img], axis=1)

print(f"รวมข้อมูลสำเร็จ! ขนาดตารางใหม่: {df_final.shape}")
print(f"จำนวนตัวแปรจากภาพถ่ายดาวเทียม {len(img_col_names)} คอลัมน์)")


Image Feature Extraction...
   - โหลดโมเดล EfficientNetB0 เรียบร้อย
   - กำลังประมวลผลภาพ 14722 ภาพ...
     > Processed 100/14722
     > Processed 200/14722
     > Processed 300/14722
     > Processed 400/14722
     > Processed 500/14722
     > Processed 600/14722
     > Processed 700/14722
     > Processed 800/14722
     > Processed 900/14722
     > Processed 1000/14722
     > Processed 1100/14722
     > Processed 1200/14722
     > Processed 1300/14722
     > Processed 1400/14722
     > Processed 1500/14722
     > Processed 1600/14722
     > Processed 1700/14722
     > Processed 1800/14722
     > Processed 1900/14722
     > Processed 2000/14722
     > Processed 2100/14722
     > Processed 2200/14722
     > Processed 2300/14722
     > Processed 2400/14722
     > Processed 2500/14722
     > Processed 2600/14722
     > Processed 2700/14722
     > Processed 2800/14722
     > Processed 2900/14722
     > Processed 3000/14722
     > Processed 3100/14722
     > Processed 3200/14722
     > Pro

In [16]:
# เตรียมข้อมูลตาราง
print("เตรียมข้อมูลตาราง")

# Drop ข้อมูลที่ไม่ใช้งานและราคาซื้อขาย
drop_cols = ['sale_price', 'name', 'id', 'lat', 'long']
existing_drop = [c for c in drop_cols if c in df_final.columns]
X = df_final.drop(columns=existing_drop)
y = np.log1p(df_final['sale_price'])

# แบ่ง Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessor
categorical_cols = ['district']
numerical_cols = [c for c in X.columns if c not in categorical_cols]

# One Hot Encoder แปลงข้อมูลตัวอักษรให้เป็นตัวเลข
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

เตรียมข้อมูลตาราง


In [17]:
# ตั้งค่า Params สำหรับ Grid search

model_params = {
    'Ridge': {
        'model': Ridge(),
        'params': {'regressor__alpha': [0.01, 1.0, 100.0]}
    },
    'KNN': {
        'model': KNeighborsRegressor(),
        'params': {'regressor__n_neighbors': [5, 10, 20], 'regressor__weights': ['uniform', 'distance']}
    },
    'RandomForest': {
        'model': RandomForestRegressor(random_state=42),
        'params': {'regressor__n_estimators': [100, 300], 'regressor__max_depth': [10, 20, None]}
    },
    'GradientBoosting': {
        'model': GradientBoostingRegressor(random_state=42),
        'params': {'regressor__n_estimators': [300, 500], 'regressor__learning_rate': [0.01, 0.05], 'regressor__max_depth': [3, 5]}
    },
    'XGBoost': {
        'model': xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1),
        'params': {
            'regressor__n_estimators': [500, 1000],
            'regressor__learning_rate': [0.01, 0.05],
            'regressor__max_depth': [5, 7]
        }
    },
    'LightGBM': {
        'model': lgb.LGBMRegressor(random_state=42, verbose=-1),
        'params': {
            'regressor__n_estimators': [500, 1000],
            'regressor__learning_rate': [0.01, 0.05],
            'regressor__num_leaves': [31, 50]
        }
    }
}


In [19]:
# Run Grid search
results = []
print("="*70)

for model_name, mp in model_params.items():
    start_time = time.time()
    
    clf = Pipeline(steps=[('preprocessor', preprocessor),
                          ('regressor', mp['model'])])
    
    grid = GridSearchCV(clf, mp['params'], cv=3, scoring='r2', n_jobs=-1)
    
    print(f"Training {model_name}...")
    try:
        grid.fit(X_train, y_train)
        
        best_model = grid.best_estimator_
        y_pred_log = best_model.predict(X_test)
        y_pred = np.expm1(y_pred_log)
        y_true = np.expm1(y_test)
        
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        
        duration = time.time() - start_time
        
        results.append({
            'Model': model_name,
            'Best R2': r2,
            'MAE (Baht)': mae,
            'MAPE (%)': mape,
            'Best Params': grid.best_params_,
            'Training Time (s)': duration
        })
        print(f"  --> Done! R2: {r2:.4f} | MAE: {mae:,.0f} | MAPE: {mape:.2f}% | Used: {duration:.1f}s")
        
    except Exception as e:
        print(f"  --> Error with {model_name}: {e}")

# แสดงผล Grid search
results_df = pd.DataFrame(results).sort_values(by='MAE (Baht)', ascending=True)

print("\n" + "="*70)
print("Leaderboard (Tabular + Satellite Images)")
print("="*70)
print(results_df[['Model', 'MAE (Baht)', 'MAPE (%)', 'Best R2', 'Training Time (s)']].to_string(index=False))

# Visualization
plt.figure(figsize=(12, 6))
sns.barplot(x='MAE (Baht)', y='Model', data=results_df, palette='viridis')
plt.title('Model Comparison (with Image Features): Mean Absolute Error')
plt.xlabel('MAE (Baht)')
plt.show()

if not results_df.empty:
    winner = results_df.iloc[0]
    print(f"Best Model: {winner['Model']}")
    print("Best Params")
    print(winner['Best Params'])

Training Ridge...
  --> Done! R2: 0.0872 | MAE: 1,413,113 | MAPE: 21.84% | Used: 4.0s
Training KNN...
  --> Done! R2: 0.8950 | MAE: 739,998 | MAPE: 12.35% | Used: 3.3s
Training RandomForest...
  --> Done! R2: 0.9350 | MAE: 648,569 | MAPE: 11.10% | Used: 186.9s
Training GradientBoosting...
  --> Done! R2: 0.9345 | MAE: 689,366 | MAPE: 11.62% | Used: 220.7s
Training XGBoost...
  --> Done! R2: 0.9390 | MAE: 650,770 | MAPE: 10.90% | Used: 48.9s
Training LightGBM...
  --> Done! R2: 0.9449 | MAE: 614,957 | MAPE: 10.48% | Used: 72.8s

Leaderboard (Tabular + Satellite Images)
           Model   MAE (Baht)  MAPE (%)  Best R2  Training Time (s)
        LightGBM 6.149569e+05 10.477958 0.944877          72.800790
    RandomForest 6.485694e+05 11.103780 0.935048         186.853217
         XGBoost 6.507700e+05 10.895423 0.939020          48.947205
GradientBoosting 6.893661e+05 11.617436 0.934538         220.660143
             KNN 7.399984e+05 12.349621 0.894958           3.288027
           Ridge 

NameError: name 'sns' is not defined

<Figure size 1200x600 with 0 Axes>